# S3-02: Tool Use 워크플로 (메시지 블록, tool_result, 멀티턴)
**Skilljar L05-L08: Handling Message Blocks / Sending Tool Results / Multi-turn / Implementation**

## 학습 목표
- `TextBlock`과 `ToolUseBlock`을 구분하여 처리한다
- `tool_result` 메시지를 올바른 형식으로 전송한다
- 완전한 Tool Use 루프 (while loop)를 구현한다
- 에러 처리를 포함한 안정적인 Tool Use 대화를 구현한다

## 사전 준비
`.env` 파일에 API 키가 설정되어 있어야 합니다.

In [ ]:
%pip install anthropic python-dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
import json

client = Anthropic()
model = "claude-sonnet-4-0"
print("설정 완료")

## 1. 메시지 블록 처리

Claude의 응답 `content`는 리스트이며, 두 종류의 블록이 포함될 수 있다:
- `TextBlock` (type="text"): 일반 텍스트
- `ToolUseBlock` (type="tool_use"): 도구 호출 요청 (id, name, input)

`stop_reason`으로 Claude의 의도를 판단한다:
- `"end_turn"`: 최종 텍스트 응답
- `"tool_use"`: 도구 호출 요청 → 결과를 돌려줘야 함

In [ ]:
# 도구 함수와 스키마 (S3_01에서 가져옴)
def get_weather(city: str) -> dict:
    """도시의 현재 날씨를 반환한다 (데모용)."""
    data = {
        "서울": {"temp": 15, "condition": "맑음", "humidity": 45},
        "부산": {"temp": 18, "condition": "구름", "humidity": 60},
    }
    return data.get(city, {"temp": 0, "condition": "알 수 없음"})

weather_tool = {
    "name": "get_weather",
    "description": "지정한 도시의 현재 날씨를 반환한다.",
    "input_schema": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "도시 이름"}
        },
        "required": ["city"]
    }
}

tool_map = {"get_weather": get_weather}

In [ ]:
# 1단계: 도구 호출 요청 받기
response = client.messages.create(
    model=model, max_tokens=1024,
    tools=[weather_tool],
    messages=[{"role": "user", "content": "서울 날씨 알려줘"}]
)

print(f"stop_reason: {response.stop_reason}")
print(f"블록 수: {len(response.content)}")

# 각 블록 분석
for i, block in enumerate(response.content):
    print(f"\n[Block {i}] type={block.type}")
    if block.type == "text":
        print(f"  text: {block.text[:100]}")
    elif block.type == "tool_use":
        print(f"  id: {block.id}")
        print(f"  name: {block.name}")
        print(f"  input: {block.input}")

## 2. tool_result 전송

도구를 실행한 후 결과를 `tool_result` 형식으로 전달해야 한다:
1. `role: "user"` — Claude 입장에서 외부 정보
2. `tool_use_id`: ToolUseBlock의 `id`와 일치
3. `content`: 실행 결과 (문자열)
4. (선택) `is_error: True` — 에러 발생 시

In [ ]:
# 2단계: 도구 실행 + tool_result 전송 + 최종 응답 받기

# messages에 전체 대화를 누적
messages = [{"role": "user", "content": "서울 날씨 알려줘"}]

# 첫 번째 API 호출
response = client.messages.create(
    model=model, max_tokens=1024,
    tools=[weather_tool],
    messages=messages
)

# assistant 응답을 messages에 추가
messages.append({"role": "assistant", "content": response.content})

# ToolUseBlock에서 함수 실행
tool_results = []
for block in response.content:
    if block.type == "tool_use":
        func = tool_map[block.name]
        result = func(**block.input)
        print(f"도구 실행: {block.name}({block.input}) -> {result}")
        
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": block.id,
            "content": json.dumps(result, ensure_ascii=False)
        })

# tool_result를 messages에 추가
messages.append({"role": "user", "content": tool_results})

# 두 번째 API 호출 — Claude가 도구 결과를 바탕으로 응답
response2 = client.messages.create(
    model=model, max_tokens=1024,
    tools=[weather_tool],
    messages=messages
)

print(f"\nstop_reason: {response2.stop_reason}")
print(f"최종 응답: {response2.content[0].text}")

## 3. 완전한 Tool Use 루프

실제 애플리케이션에서는 **while 루프**로 도구 호출을 반복 처리한다:
1. `stop_reason == "tool_use"` → 도구 실행 후 재호출
2. `stop_reason == "end_turn"` → 최종 응답 출력, 루프 종료

In [ ]:
# 완전한 Tool Use 루프 함수
def run_tool_loop(user_message: str, tools: list, tool_map: dict, system: str = None) -> str:
    """Tool Use 루프를 실행하고 최종 텍스트 응답을 반환한다."""
    messages = [{"role": "user", "content": user_message}]
    max_iterations = 5  # 무한루프 방지
    
    for i in range(max_iterations):
        params = {
            "model": model,
            "max_tokens": 2048,
            "tools": tools,
            "messages": messages
        }
        if system:
            params["system"] = system
        
        response = client.messages.create(**params)
        
        if response.stop_reason == "end_turn":
            # 최종 응답
            final_text = ""
            for block in response.content:
                if block.type == "text":
                    final_text += block.text
            return final_text
        
        elif response.stop_reason == "tool_use":
            # assistant 응답 추가
            messages.append({"role": "assistant", "content": response.content})
            
            # 각 tool_use 블록 처리
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    func = tool_map.get(block.name)
                    if func:
                        try:
                            result = func(**block.input)
                            print(f"  [Iter {i+1}] {block.name}({block.input}) -> {result}")
                            tool_results.append({
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": json.dumps(result, ensure_ascii=False)
                            })
                        except Exception as e:
                            tool_results.append({
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": f"Error: {str(e)}",
                                "is_error": True
                            })
                    else:
                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": f"Error: Unknown tool '{block.name}'",
                            "is_error": True
                        })
            
            messages.append({"role": "user", "content": tool_results})
        else:
            return f"Unexpected stop_reason: {response.stop_reason}"
    
    return "최대 반복 횟수 초과"

# 테스트
result = run_tool_loop("서울 날씨가 어때?", [weather_tool], tool_map)
print(f"\n최종 응답: {result}")

---
## 연습 1: tool_result 직접 구성

도구 호출 후 `tool_result` 메시지를 올바르게 구성하는 연습입니다.

**요구사항:**
1. 아래 도구를 등록하고 Claude에게 질문
2. ToolUseBlock을 추출하여 함수 실행
3. `tool_result`를 올바른 형식으로 구성 (tool_use_id 매칭 필수)
4. 두 번째 API 호출로 최종 응답 확인

In [ ]:
# TODO: tool_result를 직접 구성하세요

def get_material_strength(material: str, grade: str) -> dict:
    """건축 재료의 설계 강도를 반환한다."""
    data = {
        ("concrete", "24"): {"fck": 24, "Ec": 25742, "unit": "MPa"},
        ("concrete", "27"): {"fck": 27, "Ec": 27234, "unit": "MPa"},
        ("concrete", "30"): {"fck": 30, "Ec": 28608, "unit": "MPa"},
        ("rebar", "SD400"): {"fy": 400, "Es": 200000, "unit": "MPa"},
        ("rebar", "SD500"): {"fy": 500, "Es": 200000, "unit": "MPa"},
    }
    return data.get((material, grade), {"error": "데이터 없음"})

material_tool = {
    "name": "get_material_strength",
    "description": "건축 재료(콘크리트, 철근)의 설계 강도와 탄성계수를 반환한다.",
    "input_schema": {
        "type": "object",
        "properties": {
            "material": {"type": "string", "description": "재료 종류 (concrete/rebar)"},
            "grade": {"type": "string", "description": "등급 (예: 24, 27, SD400)"}
        },
        "required": ["material", "grade"]
    }
}

# TODO: 아래를 완성하세요
# 1. messages 초기화
# 2. 첫 API 호출
# 3. tool_result 구성
# 4. 두 번째 API 호출
# 5. 최종 응답 출력

In [ ]:
# ===== 정답 =====

# 1. messages 초기화
messages = [{"role": "user", "content": "콘크리트 27MPa급의 설계 강도와 탄성계수를 알려줘."}]

# 2. 첫 API 호출
resp1 = client.messages.create(
    model=model, max_tokens=1024,
    tools=[material_tool],
    messages=messages
)

print(f"1차 stop_reason: {resp1.stop_reason}")
assert resp1.stop_reason == "tool_use"

# assistant 응답 추가
messages.append({"role": "assistant", "content": resp1.content})

# 3. tool_result 구성
tool_results = []
for block in resp1.content:
    if block.type == "tool_use":
        result = get_material_strength(**block.input)
        print(f"함수 실행: {block.name}({block.input}) -> {result}")
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": block.id,  # 반드시 매칭!
            "content": json.dumps(result, ensure_ascii=False)
        })

messages.append({"role": "user", "content": tool_results})

# 4. 두 번째 API 호출
resp2 = client.messages.create(
    model=model, max_tokens=1024,
    tools=[material_tool],
    messages=messages
)

# 5. 최종 응답 출력
print(f"\n2차 stop_reason: {resp2.stop_reason}")
print(f"최종 응답: {resp2.content[0].text}")

def verify():
    assert resp2.stop_reason == "end_turn", "최종 응답은 end_turn이어야 함"
    assert any(b.type == "text" for b in resp2.content), "텍스트 블록이 있어야 함"
    assert len(tool_results) > 0, "tool_result가 생성되어야 함"
    assert "tool_use_id" in tool_results[0], "tool_use_id가 필수"
    print("모든 검증 통과!")

verify()

---
## 연습 2: 에러 처리를 포함한 Tool Use

잘못된 입력이 들어왔을 때 `is_error: True`로 에러를 전달하는 패턴을 구현하세요.

**요구사항:**
1. 0으로 나누기 등 에러를 발생시키는 도구 호출을 유도
2. `try/except`로 에러를 잡아서 `is_error: True`로 전달
3. Claude가 에러를 받고 적절한 안내 메시지를 생성하는지 확인

In [ ]:
# TODO: 에러 처리를 포함한 Tool Use 구현

In [ ]:
# ===== 정답 =====

def divide(a: float, b: float) -> dict:
    """나눗셈을 수행한다. b가 0이면 에러."""
    if b == 0:
        raise ValueError("0으로 나눌 수 없습니다.")
    return {"a": a, "b": b, "result": a / b}

divide_tool = {
    "name": "divide",
    "description": "두 수의 나눗셈을 수행한다.",
    "input_schema": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "피제수"},
            "b": {"type": "number", "description": "제수"}
        },
        "required": ["a", "b"]
    }
}

result = run_tool_loop(
    "100을 0으로 나누면 얼마야?",
    [divide_tool],
    {"divide": divide}
)
print(f"\n최종 응답: {result}")

def verify():
    assert isinstance(result, str) and len(result) > 0, "응답이 있어야 함"
    print("검증 통과: 에러 처리 후 Claude가 적절한 안내를 생성함!")

verify()

---
## 연습 3: 멀티턴 Tool Use 대화

Tool Use 루프를 대화형으로 확장하세요.

**요구사항:**
1. `run_tool_loop` 함수를 활용하여 여러 턴의 대화를 구현
2. 첫 질문의 응답을 바탕으로 후속 질문을 자동 생성
3. 전체 대화 기록을 유지하면서 도구를 반복 사용

In [ ]:
# TODO: 멀티턴 Tool Use 대화 구현

In [ ]:
# ===== 정답 =====

def run_multiturn_tool_chat(questions: list, tools: list, tool_map: dict) -> list:
    """여러 질문에 대해 멀티턴 Tool Use 대화를 수행한다."""
    messages = []
    responses = []
    
    for q_idx, question in enumerate(questions):
        print(f"\n{'='*60}")
        print(f"[턴 {q_idx+1}] 사용자: {question}")
        messages.append({"role": "user", "content": question})
        
        for i in range(5):  # 최대 5회 도구 호출
            response = client.messages.create(
                model=model, max_tokens=2048,
                tools=tools, messages=messages
            )
            
            if response.stop_reason == "end_turn":
                text = response.content[0].text if response.content else ""
                messages.append({"role": "assistant", "content": text})
                print(f"[턴 {q_idx+1}] Claude: {text[:200]}...")
                responses.append(text)
                break
            elif response.stop_reason == "tool_use":
                messages.append({"role": "assistant", "content": response.content})
                tool_results = []
                for block in response.content:
                    if block.type == "tool_use":
                        func = tool_map.get(block.name)
                        try:
                            result = func(**block.input)
                            tool_results.append({
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": json.dumps(result, ensure_ascii=False)
                            })
                        except Exception as e:
                            tool_results.append({
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": str(e),
                                "is_error": True
                            })
                messages.append({"role": "user", "content": tool_results})
    
    return responses

questions = [
    "서울 날씨가 어때?",
    "부산은?",
    "두 도시 중 더 따뜻한 곳은?"
]

results = run_multiturn_tool_chat(questions, [weather_tool], tool_map)

def verify():
    assert len(results) == 3, f"3개의 응답이 있어야 함, got {len(results)}"
    assert all(isinstance(r, str) and len(r) > 0 for r in results)
    print("모든 검증 통과! 멀티턴 Tool Use 대화 성공!")

verify()

---
## 건축공학 실습: RC 보 전단 설계 검토 워크플로

### 과제: 전단 설계 검토 도구를 Tool Use 루프로 완전히 구현하세요.

**요구사항:**
1. 전단 강도 계산 함수 (`calculate_shear_strength`) 작성
2. 도구 스키마 정의
3. `run_tool_loop`을 사용하여 완전한 검토 수행
4. Claude가 도구 결과를 해석한 최종 응답 확인

**검토 조건:**
- 보 단면: 350 x 600 mm
- fck = 27 MPa, fy = 400 MPa
- 전단보강근: D10@200 (양다리, Av = 142.6 mm2)
- 설계 전단력: Vu = 280 kN

In [ ]:
# TODO: 전단 설계 검토 도구 + Tool Use 루프 구현

In [ ]:
# ===== 정답 =====
import math

def calculate_shear_strength(b, d, fck, fy, Av, s, Vu):
    """RC 보 전단 강도 검토 (KDS 14 20 22)"""
    # 콘크리트 전단 강도
    Vc = (1/6) * math.sqrt(fck) * b * d / 1000  # kN
    # 전단 보강근 강도
    Vs = Av * fy * d / s / 1000  # kN
    # 설계 전단 강도
    phi_Vn = 0.75 * (Vc + Vs)
    # 판정
    return {
        "Vc_kN": round(Vc, 1),
        "Vs_kN": round(Vs, 1),
        "phi_Vn_kN": round(phi_Vn, 1),
        "Vu_kN": Vu,
        "DCR": round(Vu / phi_Vn, 3),
        "check": "OK" if phi_Vn >= Vu else "NG"
    }

shear_tool = {
    "name": "calculate_shear_strength",
    "description": "RC 보의 전단 강도를 KDS 14 20 22 기준으로 검토한다. Vc, Vs, phi*Vn을 계산하고 Vu와 비교 판정한다.",
    "input_schema": {
        "type": "object",
        "properties": {
            "b": {"type": "number", "description": "보 너비 (mm)"},
            "d": {"type": "number", "description": "유효 깊이 (mm)"},
            "fck": {"type": "number", "description": "콘크리트 설계기준강도 (MPa)"},
            "fy": {"type": "number", "description": "철근 항복강도 (MPa)"},
            "Av": {"type": "number", "description": "전단 보강근 단면적 (mm2, 양다리)"},
            "s": {"type": "number", "description": "전단 보강근 간격 (mm)"},
            "Vu": {"type": "number", "description": "설계 전단력 (kN)"}
        },
        "required": ["b", "d", "fck", "fy", "Av", "s", "Vu"]
    }
}

shear_tool_map = {"calculate_shear_strength": calculate_shear_strength}

final = run_tool_loop(
    "350x600 보의 전단 설계를 검토해줘. fck=27MPa, fy=400MPa, D10@200 (Av=142.6mm2), Vu=280kN",
    [shear_tool],
    shear_tool_map
)
print(f"\n최종 응답:\n{final}")

def verify():
    # 함수 직접 검증
    r = calculate_shear_strength(350, 600, 27, 400, 142.6, 200, 280)
    assert r["Vc_kN"] > 0, "Vc는 양수여야 함"
    assert r["Vs_kN"] > 0, "Vs는 양수여야 함"
    assert r["check"] in ("OK", "NG"), "판정은 OK 또는 NG"
    # 최종 응답 검증
    assert isinstance(final, str) and len(final) > 50, "최종 응답이 충분히 길어야 함"
    print(f"모든 검증 통과! Vc={r['Vc_kN']}kN, Vs={r['Vs_kN']}kN, phi*Vn={r['phi_Vn_kN']}kN, 판정: {r['check']}")

verify()